# 01 — System Preparation
Prepare a protein system for GROMACS MD simulation.
Supports both soluble (aqueous) and membrane-bound proteins.

## Prerequisites
- GROMACS 2026.3 (`conda activate gromacs_env`)
- A PDB file (e.g. from RCSB PDB, AlphaFold DB, or BioEmu)
- Force field: CHARMM36m (recommended for proteins)
- Water model: TIP3P for soluble, CHARMM-GUI membrane builder recommended for membrane

### Configuration — Edit these values

In [ ]:
import os
import subprocess
import warnings

# ── User parameters ──────────────────────────────────────────
PDB_FILE = "data/input.pdb"         # Input protein structure
PROTEIN_NAME = "my_protein"           # Name for output files
FORCE_FIELD = "amber14sb"    # Force field (charmm36m, amber99sb-ildn, etc.)
WATER_MODEL = "tip3p"                # Water model
BOX_TYPE = "dodecahedron"            # Box shape: cubic, dodecahedron, octahedron
BOX_DISTANCE = 1.2                   # Distance from protein to box edge (nm)
ION_CONC = 0.15                      # NaCl concentration (M)

# ── Membrane parameters (ignored for soluble proteins) ───────
MEMBRANE_SIMULATION = False          # Set True for membrane proteins
MEMBRANE_LIPID = "POPC"              # Lipid type
MEMBRANE_AREA = 8.0                  # Approximate area per lipid (nm^2)

# ── Output directory ─────────────────────────────────────────
OUTPUT_DIR = f"data/{PROTEIN_NAME}_prep"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs('data', exist_ok=True)

print(f"Protein: {PROTEIN_NAME}")
print(f"Input PDB: {PDB_FILE}")
print(f"Force field: {FORCE_FIELD}")
print(f"Simulation type: {'MEMBRANE' if MEMBRANE_SIMULATION else 'SOLUBLE'}")
warnings.filterwarnings('ignore')

## Step 1: Check input structure
Clean the PDB — remove non-protein residues, alternate conformations, and HETATMs.

In [ ]:
def run_gmx(cmd, desc=""):
    """Run a GROMACS command and print output."""
    print(f"  [{desc}]" if desc else f"  $ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  WARNING: exit code {result.returncode}")
        print(result.stderr[-500:] if result.stderr else "")
    return result

In [ ]:
import mdtraj as md

# Load and inspect the PDB
pdb_path = PDB_FILE  # Use as-is or download from RCSB below
traj = md.load(pdb_path)
top = traj.topology
print(f"Frames: {traj.n_frames}")
print(f"Atoms: {traj.n_atoms}")
print(f"Residues: {top.n_residues}")
print(f"Chains: {top.n_chains}")
print(f"\nResidues: {[str(r) for r in top.residues[:5]]}...")
print(f"Box vectors: {traj.unitcell_vectors}")

### Optional: Download a PDB from RCSB or AlphaFold DB

In [ ]:
# Uncomment to download from RCSB PDB
# PDB_ID = "7rei"  # Example: CB1 receptor
# url = f"https://files.rcsb.org/download/{PDB_ID}.pdb"
# !wget -q {url} -O {PDB_FILE}
# print(f"Downloaded {PDB_ID}")

# Uncomment to download from AlphaFold DB (UniProt ID)
# UNIPROT_ID = "P21554"  # Example: human CB1
# url = f"https://alphafold.ebi.ac.uk/files/AF-{UNIPROT_ID}-F1-model_v4.pdb"
# !wget -q {url} -O {PDB_FILE}
# print(f"Downloaded AlphaFold model for {UNIPROT_ID}")

## Step 2: Prepare structure for GROMACS
Remove water and HETATMs, then generate topology.

In [ ]:
prep_dir = OUTPUT_DIR

# Remove crystallographic water and HETATMs
run_gmx(f"gmx pdb2gmx -f {pdb_path} -o {prep_dir}/processed.gro -p {prep_dir}/topol.top "
        f"-ignh -ff {FORCE_FIELD} -water {WATER_MODEL}",
        desc="pdb2gmx: generate topology")

# Check the topology
if os.path.exists(f"{prep_dir}/topol.top"):
    with open(f"{prep_dir}/topol.top") as f:
        top_content = f.read()
    print("\nTopology file written.")
    # Count molecules
    for line in top_content.split('\n'):
        if 'Protein_chain' in line or 'Protein' in line:
            print(f"  {line.strip()}")

## Step 3: Define simulation box

In [ ]:
run_gmx(f"gmx editconf -f {prep_dir}/processed.gro -o {prep_dir}/box.gro -c "
        f"-bt {BOX_TYPE} -d {BOX_DISTANCE}",
        desc="editconf: define box")

# Check box size
traj_box = md.load(f"{prep_dir}/box.gro")
print(f"\nBox vectors: {traj_box.unitcell_vectors[0]}")

## Step 4: Solvate
For **soluble proteins**: fill box with water.
For **membrane proteins**: embed in lipid bilayer (requires external tool or pre-equilibrated membrane).

In [ ]:
if not MEMBRANE_SIMULATION:
    # Solvate with water
    run_gmx(f"gmx solvate -cp {prep_dir}/box.gro -cs spc216.gro -o {prep_dir}/solvated.gro "
            f"-p {prep_dir}/topol.top",
            desc="solvate: add water")
    
    # Add ions
    run_gmx(f"gmx grompp -f {prep_dir}/mdp/minim.mdp -c {prep_dir}/solvated.gro -p {prep_dir}/topol.top "
            f"-o {prep_dir}/ions.tpr -maxwarn 1",
            desc="grompp: prepare for ions")
    run_gmx(f"echo -e 'SOL' | gmx genion -s {prep_dir}/ions.tpr -o {prep_dir}/solvated_ions.gro "
            f"-p {prep_dir}/topol.top -pname NA -nname CL -neutral -conc {ION_CONC}",
            desc="genion: add NaCl")
    
    final_gro = f"{prep_dir}/solvated_ions.gro"
    print(f"\nSoluble system ready: {final_gro}")
else:
    print("\n[!] Membrane protein simulation.")
    print("  Use CHARMM-GUI Membrane Builder (https://charmm-gui.org) to:")
    print("  1. Orient the protein in the membrane")
    print("  2. Add lipid bilayer")
    print("  3. Solvate and ionize")
    print("  4. Download the GROMACS input files")
    print("\n  Alternative: Use the insane.py script by Wassenaar et al.")
    print("  Place the output files in the preparation directory.\n")
    final_gro = f"{prep_dir}/step5_charmm2gmx.pdb"  # Typical CHARMM-GUI output
    print(f"  Expected input: {final_gro}")

## Step 5: Energy minimization (steepest descent)

In [ ]:
# Create MDP files for minimization
import textwrap

minim_mdp = textwrap.dedent("""\
; Energy minimization
integrator  = steep
nsteps     = 5000
nstlog     = 100
nstenergy  = 100
nstxout    = 0
emtol      = 1000.0
emstep     = 0.01
cutoff-scheme = Verlet
ns_type    = grid
nstlist    = 20
rlist      = 1.2
coulombtype = PME
rcoulomb   = 1.2
vdwtype    = cut-off
rvdw       = 1.2
DispCorr   = no
pbc        = xyz
; Constraints
constraints             = h-bonds
constraint-algorithm    = lincs
lincs-order             = 4
""")

os.makedirs(f"{prep_dir}/mdp", exist_ok=True)
with open(f"{prep_dir}/mdp/minim.mdp", 'w') as f:
    f.write(minim_mdp)
print("MDP for energy minimization written.")

In [ ]:
run_gmx(f"gmx grompp -f {prep_dir}/mdp/minim.mdp -c {final_gro} -p {prep_dir}/topol.top "
        f"-o {prep_dir}/em.tpr -pp {prep_dir}/processed.top -po {prep_dir}/mdp/minim_out.mdp -maxwarn 1",
        desc="grompp: prepare EM")

run_gmx(f"gmx mdrun -v -deffnm {prep_dir}/em -s {prep_dir}/em.tpr -gpu_id 0 -nb gpu -pme cpu",
        desc="mdrun: energy minimization")

In [ ]:
# Check minimization result
import numpy as np

if os.path.exists(f"{prep_dir}/em.edr"):
    result = subprocess.run(f"echo -e '10\n0\n' | gmx energy -f {prep_dir}/em.edr -o {prep_dir}/em_potential.xvg",
                            shell=True, capture_output=True, text=True)
    print(result.stdout[-300:] if result.stdout else "")
    
    # Parse final potential energy
    import pandas as pd
    data = pd.read_csv(f"{prep_dir}/em_potential.xvg", comment='#', comment='@',
                       delim_whitespace=True, header=None, names=['time','pot'])
    print(f"\nFinal potential energy: {data['pot'].values[-1]:.2f} kJ/mol")
    print(f"Initial potential energy: {data['pot'].values[0]:.2f} kJ/mol")
    
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.plot(data['pot'])
    plt.xlabel('Step')
    plt.ylabel('Potential energy (kJ/mol)')
    plt.title('Energy minimization')
    plt.grid(alpha=0.3)
    plt.savefig(f"{prep_dir}/em_convergence.png", dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Plot saved: {prep_dir}/em_convergence.png")

## Summary
The prepared system is ready for equilibration. Key output files:
- `{prep_dir}/em.gro` — minimized structure
- `{prep_dir}/topol.top` — topology
- `{prep_dir}/em.tpr` — run input file

Proceed to `02_running.ipynb` for equilibration and production MD.